In [1]:
# Installer les dépendances indiquées par le projet.
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 51.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 76.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.1/31.1 MB 71.2 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 61.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 73.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 24.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 72.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.8/39.8 MB 86.5 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 65.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46/46 [streamlit]treamlit]ltair]ema]s]art]

[notice] A new release of pip is avai

In [2]:
# Bibliothèques utilisées pour collecter et organiser les documents.
import praw
import pandas as pd
import urllib.request
import urllib.parse
import xmltodict
import ssl
import certifi

from datetime import datetime, timezone

# Classes du projet.
from classes.Document import RedditDocument, ArxivDocument
from classes.Corpus import Corpus
from factories.DocumentFactory import DocumentFactory

In [4]:
from pathlib import Path
from classes.Corpus import Corpus

# Choisir l'origine des données.
mode = input(
    "1 : charger le corpus sauvegardé\n"
    "2 : collecter de nouveaux documents\n"
    "Votre choix : "
).strip()

if mode not in {"1", "2"}:
    raise ValueError("Choisissez 1 ou 2.")

corpus = Corpus("Animaux")

if mode == "1":
    chemin = Path("corpus_objets.csv")

    if not chemin.is_file():
        raise FileNotFoundError(
            "Aucun corpus sauvegardé. Relancez cette cellule "
            "et choisissez 2 pour collecter."
        )

    corpus.load(chemin)
    print("Corpus chargé :", corpus)
else:
    print("Mode collecte sélectionné.")

Mode collecte sélectionné.


In [5]:
if mode == "2":
    from collecte import collecter_documents

    # Connexion Reddit.
    reddit = praw.Reddit("projet")

    # Collecter les textes et conserver leurs métadonnées.
    docs, donnees_documents, nb_reddit, nb_arxiv = collecter_documents(
        reddit=reddit,
        theme_reddit="Cats",
        theme_arxiv="animal",
        taille_reddit=100,
        taille_arxiv=100,
    )

    # Afficher un aperçu court.
    for i, texte in enumerate(docs[:3]):
        print(f"\nDocument {i} : {texte[:200]}...")


Champs Reddit utilisés :
title, author, created_utc, permalink, selftext, num_comments
Exemple de titre : Letting users remove posts.

Champs arXiv : ['id', 'title', 'updated', 'link', 'summary', 'category', 'published', 'arxiv:comment', 'arxiv:primary_category', 'arxiv:journal_ref', 'author']

Documents Reddit : 69
Documents arXiv : 100
Total avant filtrage : 169

Document 0 : Hello everyone,   As you are aware, on reddit moderators approve and remove content and subreddit subscribers can report to us.  Today we'd like to start an experiment here on r/Cats where we will giv...

Document 1 : I don't know if you can tell from the photos, but she is the neediest most *lovable* cat I've ever met in my life!...

Document 2 : Hi Cat Crew!  Just wanted to share my recent adoptions. Nameless at the moment but have some strong contenders. I went for one… came out with a bonded pair. Couldn’t resist the opportunity to show the...


In [6]:
if mode == "2":

    import pandas as pd

    # La collecte ajoute d'abord Reddit, puis arXiv.
    sources = ["reddit"] * nb_reddit + ["arxiv"] * nb_arxiv

    df = pd.DataFrame({
        "id": range(len(docs)),
        "texte": docs,
        "source": sources
    })

    df.to_csv("corpus.csv", sep="\t", index=False, encoding="utf-8")

    print(f"Sauvegarde terminée : {len(df)} documents.")
    print(df["source"].value_counts())

Sauvegarde terminée : 169 documents.
source
arxiv     100
reddit     69
Name: count, dtype: int64


In [7]:
#Réalise la première sauvegarde des textes des deux sources.
if mode == "2":
    df_charge = pd.read_csv("corpus.csv", sep="\t", encoding="utf-8")

    assert df_charge.equals(df), "Le tableau rechargé est différent !"

    print(f"Rechargement vérifié : {len(df_charge)} documents identiques.")

Rechargement vérifié : 169 documents identiques.


In [8]:
# Compter les mots et les phrases après chargement ou collecte.
print("Nombre de documents :", len(corpus.documents))

for identifiant, document in corpus.documents.items():
    texte = document.texte

    nb_mots = len(texte.split())
    nb_phrases = len([
        phrase
        for phrase in texte.split(".")
        if phrase.strip()
    ])

    print(
        f"Document {identifiant} : "
        f"{nb_mots} mots, {nb_phrases} phrases"
    )

Nombre de documents : 154
Document 0 : 534 mots, 29 phrases
Document 1 : 378 mots, 24 phrases
Document 2 : 82 mots, 3 phrases
Document 3 : 29 mots, 3 phrases
Document 4 : 1051 mots, 47 phrases
Document 5 : 31 mots, 2 phrases
Document 6 : 154 mots, 7 phrases
Document 7 : 109 mots, 7 phrases
Document 8 : 256 mots, 17 phrases
Document 9 : 96 mots, 5 phrases
Document 10 : 236 mots, 15 phrases
Document 11 : 48 mots, 3 phrases
Document 12 : 255 mots, 17 phrases
Document 13 : 110 mots, 5 phrases
Document 14 : 45 mots, 3 phrases
Document 15 : 425 mots, 29 phrases
Document 16 : 22 mots, 1 phrases
Document 17 : 345 mots, 22 phrases
Document 18 : 167 mots, 10 phrases
Document 19 : 104 mots, 5 phrases
Document 20 : 282 mots, 18 phrases
Document 21 : 173 mots, 10 phrases
Document 22 : 26 mots, 2 phrases
Document 23 : 101 mots, 7 phrases
Document 24 : 427 mots, 34 phrases
Document 25 : 24 mots, 3 phrases
Document 26 : 32 mots, 5 phrases
Document 27 : 111 mots, 17 phrases
Document 28 : 19 mots, 2 phr

In [9]:
if mode == "2":
    nb_avant = len(df)

    df = df[df["texte"].str.len() >= 100].copy()
    df = df.reset_index(drop=True)

    # Réattribuer des identifiants consécutifs.
    df["id"] = range(len(df))

    # Mettre à jour la liste des textes.
    docs = df["texte"].tolist()

    print("Documents supprimés :", nb_avant - len(df))
    print("Documents conservés :", len(docs))

Documents supprimés : 15
Documents conservés : 154


In [10]:
# Sauvegarder les textes filtrés après une nouvelle collecte.
if mode == "2":
    df.to_csv(
        "corpus.csv",
        sep="\t",
        index=False,
        encoding="utf-8",
    )

    print("Corpus filtré sauvegardé.")

Corpus filtré sauvegardé.


In [11]:
if mode == "2":
    # Construire le corpus avec les docs.
    corpus = Corpus("Animaux")

    corpus.clear()

    for donnees in donnees_documents:
        if len(donnees["texte"]) < 100:
            continue

        attributs = donnees.copy()
        source = attributs.pop("source")

        document = DocumentFactory.create(
            source=source,
            **attributs
        )

        corpus.add(document)

    assert len(corpus.documents) == len(df)

    assert [
        document.texte
        for document in corpus.documents.values()
    ] == df["texte"].tolist()

    print(corpus)

Corpus 'Animaux' : 154 document(s), 149 auteur(s)


In [12]:

# Afficher les documents réels par titre et par date.
print("Par titre :")
corpus.afficher_par_titre(5)

print("\nPar date, les plus récents d'abord :")
corpus.afficher_par_date(5)

Par titre :
[Reddit] 3 months of Pepe, my rescued kitten (32 commentaires)
[arXiv] 4D-Animal: Freely Reconstructing Animatable 3D Animals from Videos
[Reddit] A mom cat dropped 4 kittens off at my door six months ago and this little guy has imprinted on me 🥹 (16 commentaires)
[arXiv] A Novel Dataset for Keypoint Detection of quadruped Animals from Images
[arXiv] A Review on Coarse to Fine-Grained Animal Action Recognition

Par date, les plus récents d'abord :
2026-11-07T08:50:05+00:00 — [Reddit] Elderly cat (2 commentaires)
2026-11-07T08:32:08+00:00 — [Reddit] Older cat grooms spot until shes bleeding any will only stop with grooming and vets have been no help for past 3 years (30 commentaires)
2026-11-07T08:28:44+00:00 — [Reddit] The first and the last. (8 commentaires)
2026-11-07T08:23:06+00:00 — [Reddit] Cat attack on the carpet 😼 (2 commentaires)
2026-11-07T07:57:40+00:00 — [Reddit] Tribute to Mini: 13.05.2019 - 04.11.2026 (23 commentaires)


In [13]:
# Vérifier le polymorphisme et afficher la source de chaque doc.
if mode == "2":
    for identifiant, document in corpus.documents.items():
        assert isinstance(document, (RedditDocument, ArxivDocument))
        print(identifiant, document.getType(), ":", document)

    print("\nNombre de documents par source :")
    print(pd.Series([
        document.getType()
        for document in corpus.documents.values()
    ]).value_counts())

0 reddit : [Reddit] Letting users remove posts. (164 commentaires)
1 reddit : [Reddit] Baby Honeysuckle versus 4 year old Honeysuckle (200 commentaires)
2 reddit : [Reddit] My new feline friends! (56 commentaires)
3 reddit : [Reddit] Had to put lil Vicky down last night😭 (93 commentaires)
4 reddit : [Reddit] Tribute to Dillon - Feb 18 2013 - Nov 6 2026 (105 commentaires)
5 reddit : [Reddit] Willow is so freaking cute !!! (51 commentaires)
6 reddit : [Reddit] Tribute to Mini: 13.05.2019 - 04.11.2026 (23 commentaires)
7 reddit : [Reddit] I feel like I've made a mistake changing the room I play on my PC in. Sad cat noises outside my room 😪 (33 commentaires)
8 reddit : [Reddit] 3 months of Pepe, my rescued kitten (32 commentaires)
9 reddit : [Reddit] Maggie the survivor (22 commentaires)
10 reddit : [Reddit] My cat is the same color as our bathroom door mat lol!!! (33 commentaires)
11 reddit : [Reddit] So I was a little awkward in high school. (54 commentaires)
12 reddit : [Reddit] Dyrone 

In [14]:
# Mémoriser les données avant de recharger le corpus unique.
if mode == "2":

    avant = [
        (type(document).__name__, vars(document).copy())
        for document in corpus.documents.values()
    ]

    corpus.save("corpus_objets.csv")
    corpus.load("corpus_objets.csv")

    apres = [
        (type(document).__name__, vars(document).copy())
        for document in corpus.documents.values()
    ]

    assert avant == apres

    print("Sauvegarde vérifiée : types et attributs identiques.")
    print(corpus)

Sauvegarde vérifiée : types et attributs identiques.
Corpus 'Animaux' : 154 document(s), 149 auteur(s)


In [15]:
# Vérifier qu'il y a pas de deoublons.
if mode == "2":
    autre_reference = Corpus("Autre nom")

    assert autre_reference is corpus

    print("Singleton vérifié :", autre_reference is corpus)

Singleton vérifié : True


In [16]:
# Voir les stats d'un auteur réel du corpus des docs.
print("Auteurs disponibles :")
for nom in corpus.authors:
    print("-", nom)

nom = input("\nNom de l'auteur : ").strip()
auteur = corpus.authors.get(nom)

if auteur is None:
    print("Auteur introuvable.")
else:
    taille_moyenne = (
        sum(len(document.texte) for document in auteur.production.values())
        / auteur.nb_docs
        if auteur.nb_docs
        else 0
    )

    print("Nombre de documents :", auteur.nb_docs)
    print(
        "Taille moyenne :",
        round(taille_moyenne, 1),
        "caractères"
    )

Auteurs disponibles :
- Merari01
- babysuckle
- Canal_faun
- maxiepad15
- TheRealMallow64
- BonolotaSen23
- ComeOnIWantUsername
- Inevitable_Gain8296
- ibizamik
- anotherlovelysunrise
- arimissx
- sea_bird
- _bessica_
- Quick-Ad-3378
- tablesexual
- Isitwhenip
- NaStK14
- itara8
- Flaky_Wrongdoer_7474
- myopinionstinks
- jackw91
- Bees_and_Teas
- TheRainbowCheese
- Tough-Opportunity910
- Any-Sheepherder-1821
- veregwen
- Taytay3341
- CannonHumper
- laboratory_equipment
- GaiaMoore
- Own_Oil_7719
- silentgraywarden
- NextSalary1414
- 8bitVeg
- shhbaby_isok
- OliveOnPizza
- ResearchingRabbit
- Delicious-Bicycle483
- Jaded_b3rry
- GottGobbleToo
- arcbelial
- Awkward-Initiative27
- QueenObsidian83
- esoterika81
- Groovy_Coconut77
- UbiquitousMintCookie
- LBPMar
- Voidandverse
- Technical_Car3811
- AdHonest945
- Novice_KerbalDude123
- gdb7
- No-Fun-9576
- Blainessad
- Andrea Cavagna
- David Lusseau
- Xun Long Ng
- Xuyi Hu
- Michele Ballerini
- Eklavya Sarkar
- Levi Cai
- Stefan Schneider
- 

In [17]:
import importlib
import classes.Corpus as corpus_module

importlib.reload(corpus_module)

corpus = corpus_module.Corpus("Animaux")
corpus.load("corpus_objets.csv") 
print(len(corpus.documents), "document(s)")
print(corpus.search("cat")[:3])

154 document(s)
["   Thank you for reading, and thank you all for making this subreddit a great place to be I don't know if you can tell from the photos, but she is the neediest most *lovable* cat I've ever met in my life! Hi Cat Crew!  Just wanted to share my recent adoptions.", '   Any other blind kitten friends or recent adoptions out there? Please share ♥️ I loved this cat to death.', '  Recently I decided to change it up and set up my gaming station in the unused guest room and keep it cat free.']


In [18]:
import importlib
import classes.SearchEngine as se_module

importlib.reload(se_module)

moteur = se_module.SearchEngine(corpus)
print(moteur.mat_TFxIDF.shape)
moteur.search("cat", n=5)

Construction de la matrice: 100%|██████████| 154/154 [00:00<00:00, 21924.67it/s]

(154, 4657)


,titre,auteur,source,score
0,"3 months of Pepe, my rescued kitten",ibizamik,reddit,0.2188
1,My brother’s bestest man Django,AdHonest945,reddit,0.1528
2,My Bonded Threesome,gdb7,reddit,0.1347
3,tribute to My beloved cat .,itara8,reddit,0.1313
4,Baby Honeysuckle versus 4 year old Honeysuckle,babysuckle,reddit,0.1227
